### A3.3 Multiple testing y aprendizaje por refuerzo


Video demostrativo del aprendizaje por refuerzo de un agente.
Primero, se instala la librería `gymnasium`, `xvfb`, `ffmpeg` y `xvfb` para poder ejecutar simulaciones y además guardar las cosas en video porque se está trabajando en Google Colab.



In [1]:
!pip install gymnasium[all]
!apt-get install -y xvfb ffmpeg
!pip install pyvirtualdisplay

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.16).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


Se inicializa el display

In [2]:
import gymnasium as gym
import numpy as np
import os
from pyvirtualdisplay import Display
from gymnasium.wrappers import RecordVideo

display = Display(visible=0, size=(1400, 900))
display.start()

Luego se crea el ambiente. Se usa el de `FrozenLake-v1` que es un lago con pedazos de hielo (que sí son seguros), también hay agujeros (si cae ahí pierde) y hay una meta, como un regalo que cuando el personaje llegue ahí, gana. Da la opción de elegir entre que si el hielo es resbaloso o no. En este caso se eligió que sí fuera resbaloso, lo cual implica que aunque el agente quiera moverse a lado, puede que por esta característica no se mueva realmente para ese lado.





In [20]:
env = gym.make("FrozenLake-v1", render_mode="rgb_array", is_slippery=True)

Se establecen la cantidad de acciones posibles de acuerdo a este ecosistema y el tamaño.
Luego se crea la tabla Q la cual se inicia llenándola con 0, con dimensiones (estados y acciones). Esto se hace para que después el agente pueda ir explorando, recibiendo recompensas y vaya llenando esta tabla, actualizándola con la mejor acción para cada estado.

In [21]:
state_size = env.observation_space.n
action_size = env.action_space.n

q_table = np.zeros((state_size, action_size))

Se definen los siguientes hiperparámetros:
- learning rate = 0.1 (entre más bajo va más lento pero es más estable. Con un valor alto aprende rápido pero puede ser más inestable).
- discount factor = 0.99 (para maximizar las recompensas futuras en lugar de las recompensas inmediatas)


- epsilon = 1 (que es la probabilidad de explorar acciones aleatorias, al principio explora el 100% del tiempo)
- epsilon decay = 0.999 (lo que reduce epsilon poco a poco en cada episodio)
- epsilon min = 0.01 (siempre habrá un mínimo de exploración)


Luego se establecen 100000 episodios, que es la cantidad que el agente juega el juego para ir aprendiendo nuevas cosas. Como se tiene activado que sí sea resbaloso, el aprendizaje puede ser complicado, por eso se eligió un gran cantidad de episodios.
Luego se establecen "checkpoints" para poder guardar el progreso de cómo va aprendiendo.





In [22]:
alpha = 0.1
gamma = 0.99

epsilon = 1.0
epsilon_decay = 0.999
epsilon_min = 0.01

episodes = 100000
checkpoints = [0, 1000, 10000, 50000, 99999]

saved_q_tables = {}

Luego se inicia un for loop para el aprendizaje del agente, repitiendo el entrenamiento la cantidad de veces que se estableció anteriormente. El agente sigue jugando hasta que el episodio acabe. Se agrega una decisión para ver si sigue explorando o se explota. Con la probabilidad de epsilon, se explora con un acción aleatoria. Si no, se elige la mejor acción ya testeada.


También se agrega una línea de código para ir modificando epsilón conforme avanzan los episodios. Además de que se agrega otra en donde se establece que cuando el episodio sea el número que se establecieron para los checkpoints, se guarde.


Se va mostrando el progreso cada 200 episodios.





In [23]:
for episode in range(episodes):
    state, _ = env.reset()
    done = False

    while not done:
        if np.random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table[state])

        new_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        q_table[state, action] += alpha * (
            reward + gamma * np.max(q_table[new_state]) - q_table[state, action]
        )

        state = new_state

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if episode in checkpoints:
        saved_q_tables[episode] = q_table.copy()

    if episode % 200 == 0:
        print(f"Episode {episode}, epsilon: {epsilon:.3f}")

Episode 0, epsilon: 0.999
Episode 200, epsilon: 0.818
Episode 400, epsilon: 0.670
Episode 600, epsilon: 0.548
Episode 800, epsilon: 0.449
Episode 1000, epsilon: 0.367
Episode 1200, epsilon: 0.301
Episode 1400, epsilon: 0.246
Episode 1600, epsilon: 0.202
Episode 1800, epsilon: 0.165
Episode 2000, epsilon: 0.135
Episode 2200, epsilon: 0.111
Episode 2400, epsilon: 0.091
Episode 2600, epsilon: 0.074
Episode 2800, epsilon: 0.061
Episode 3000, epsilon: 0.050
Episode 3200, epsilon: 0.041
Episode 3400, epsilon: 0.033
Episode 3600, epsilon: 0.027
Episode 3800, epsilon: 0.022
Episode 4000, epsilon: 0.018
Episode 4200, epsilon: 0.015
Episode 4400, epsilon: 0.012
Episode 4600, epsilon: 0.010
Episode 4800, epsilon: 0.010
Episode 5000, epsilon: 0.010
Episode 5200, epsilon: 0.010
Episode 5400, epsilon: 0.010
Episode 5600, epsilon: 0.010
Episode 5800, epsilon: 0.010
Episode 6000, epsilon: 0.010
Episode 6200, epsilon: 0.010
Episode 6400, epsilon: 0.010
Episode 6600, epsilon: 0.010
Episode 6800, epsilon

Aquí se graba el agente. Toma la tabla Q, con los episodios establecidos como checkpoints, lo ejecuta usando la mejor acción y lo guarda como video.

In [24]:
def record_video(q_table, name):
    env = gym.make("FrozenLake-v1", render_mode="rgb_array", is_slippery=True)
    env = RecordVideo(env, video_folder="videos", name_prefix=name)

    state, _ = env.reset()
    done = False

    while not done:
        action = np.argmax(q_table[state])
        state, _, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

    env.close()

Aquí se guardan todos los videos que se crearon por episodio, recorriendo el progreso de aprendizaje del agente.

In [25]:
os.makedirs("videos", exist_ok=True)

for ep, q in saved_q_tables.items():
    print(f"Recording episode {ep}")
    record_video(q, f"episode_{ep}")

  logger.warn(



Recording episode 0
Recording episode 1000
Recording episode 10000
Recording episode 50000
Recording episode 99999


Con este siguiente bloque se muestran los videos, sin tener que descargarlos. Sin embargo, estos son los videos por cada checkpoint, aún no están unidos.

In [26]:
from IPython.display import HTML, display
from base64 import b64encode
import glob

def show_video(path):
    mp4 = open(path,'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML(f'<video width=400 controls><source src="{data_url}" type="video/mp4"></video>')

video_files = glob.glob("videos/*.mp4")

for v in video_files:
    display(show_video(v))

Luego se importan estas 2 librerías para poder modificar los videos que ya se generaron y poder unirlos.

In [27]:
!apt-get install -y imagemagick
!pip install moviepy

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
imagemagick is already the newest version (8:6.9.11.60+dfsg-1.3ubuntu0.22.04.5).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


En este bloque de código que sigue se unen todos los fragmentos de los diferentes episodios en un solo video. Además, se agrega un título con el número de episodios que se está reproduciendo para tener un mejor entendimiento de lo que está pasando.





In [28]:
import moviepy.config
import os

policy_file = "/etc/ImageMagick-6/policy.xml"
if os.path.exists(policy_file):
    with open(policy_file, 'r') as f:
        content = f.read()

    new_content = content.replace(
        '<policy domain="path" rights="none" pattern="@*"/>',
        '<policy domain="path" rights="read|write" pattern="@*"/>'
    )

    if new_content == content:
        new_content = content.replace(
            '<policy domain="delegate" rights="none" pattern="ephemeral"/>',
            '<policy domain="delegate" rights="read|write" pattern="ephemeral"/>'
        )

    if new_content != content:
        with open(policy_file, 'w') as f:
            f.write(new_content)
        print("ImageMagick policy updated successfully.")
    else:
        print("Could not find specific ImageMagick policy to update. Proceeding anyway.")


!which convert
!convert --version


if os.path.exists("/usr/bin/convert"):
    moviepy.config.change_settings({"IMAGEMAGICK_BINARY": "/usr/bin/convert"})
else:
    print("Warning: /usr/bin/convert not found. MoviePy might fail.")

from moviepy.editor import VideoFileClip, TextClip, CompositeVideoClip, concatenate_videoclips

def create_titled_clip(video_path, episode_num):
    clip = VideoFileClip(video_path)
    txt_clip = TextClip(f"Episode {episode_num}", fontsize=40, color='black', bg_color='transparent', font='Montserrat-Black')
    txt_clip = txt_clip.set_duration(clip.duration).set_pos(('center', 'bottom'))
    final_clip = CompositeVideoClip([clip, txt_clip])
    return final_clip


video_files_filtered = [f for f in video_files if "episode_" in f and "combined_" not in f]

video_files_filtered.sort(key=lambda x: int(x.split('episode_')[1].split('-')[0]))

clips = []
for video_file in video_files_filtered:
    episode_num = int(video_file.split('episode_')[1].split('-')[0])
    titled_clip = create_titled_clip(video_file, episode_num)
    clips.append(titled_clip)

final_clip = concatenate_videoclips(clips)
final_video_path = "videos/combined_frozenlake_episodes.mp4"
final_clip.write_videofile(final_video_path, fps=env.metadata['render_fps'])

Could not find specific ImageMagick policy to update. Proceeding anyway.
/usr/bin/convert
Version: ImageMagick 6.9.11-60 Q16 x86_64 2021-01-25 https://imagemagick.org
Copyright: (C) 1999-2021 ImageMagick Studio LLC
License: https://imagemagick.org/script/license.php
Features: Cipher DPC Modules OpenMP(4.5) 
Delegates (built-in): bzlib djvu fftw fontconfig freetype heic jbig jng jp2 jpeg lcms lqr ltdl lzma openexr pangocairo png tiff webp wmf x xml zlib
Moviepy - Building video videos/combined_frozenlake_episodes.mp4.
Moviepy - Writing video videos/combined_frozenlake_episodes.mp4



Moviepy - Done !
Moviepy - video ready videos/combined_frozenlake_episodes.mp4


Por último, se muestra el video final con todos los episodios que se establecieron anteriormente como checkpoints para ver el progreso de aprendizaje del agente.

In [29]:
display(show_video(final_video_path))